# 03 - Dependency graph and provisional risk score

**Purpose:** Show every dependency edge with provenance (OBSERVED / DECLARED / INFERRED), conflicts, cycles, the additive risk score and its sensitivity.

**Inputs:** cryptoshift.planning (demo configs + synthetic descriptor), planning_config.json

**Outputs:** edge table, graph figure, score table, sensitivity summary

**Status labels:** Graph VERIFIED on SYNTHETIC inputs. Risk score PROVISIONAL and UNVALIDATED (weights are assumptions).

TEST/demo context: the demo bank in `demo_bank_app/` is SYNTHETIC.

In [ ]:
from pathlib import Path
import sys, json
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src" / "cryptoshift").is_dir())
for p in (str(ROOT / "src"), str(ROOT)):
    if p not in sys.path:
        sys.path.insert(0, p)
DERIVED = ROOT / "scratch" / "derived" / "member3"
DERIVED.mkdir(parents=True, exist_ok=True)
# This notebook reads no credentials. If a future cell needs one, read it from an environment variable at runtime
# and never display it.

In [ ]:
import pandas as pd
from cryptoshift.planning.planner import build_problem, sensitivity
p = build_problem()
pd.DataFrame(p['edges'])[['upstream','downstream','provenance','file','line','reason']]

In [ ]:
ga = p['graph_analysis']; print('cycles:', ga['cycles'] or 'none'); print('edge counts:', ga['edge_counts'])
pd.DataFrame(ga['conflicts'])

In [ ]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt, networkx as nx
g = nx.DiGraph(); g.add_nodes_from(p['services'])
colors = {'OBSERVED':'tab:green','DECLARED':'tab:gray','INFERRED':'tab:orange'}
for e in p['edges']: g.add_edge(e['upstream'], e['downstream'], color=colors[e['provenance']])
plt.figure(figsize=(7,4)); pos = nx.spring_layout(g, seed=3)
nx.draw_networkx(g, pos, edge_color=[d['color'] for *_, d in g.edges(data=True)], node_color='lightsteelblue', font_size=8, arrowsize=15)
plt.title('upstream -> downstream (green OBSERVED, gray DECLARED)'); plt.axis('off'); plt.savefig(DERIVED / 'dependency_graph.png', dpi=120)

In [ ]:
pd.DataFrame([{'service': s, 'score': v['risk']['score'], **v['risk']['contributions']} for s, v in p['services'].items()]).sort_values('score', ascending=False)

In [ ]:
sa = sensitivity(p)
print(f"rank changed {sa['rank_changed_runs']}/{sa['n_runs']}, schedule changed {sa['schedule_changed_runs']}/{sa['n_runs']}, min Kendall tau {sa['min_kendall_tau']}")
pd.DataFrame(sa['runs'])[['weight','factor','rank_changed','kendall_tau','schedule_changed']]